In [ ]:
import numpy as np
import numpy.typing as npt
from pathlib import Path
from astropy.time import Time
from sorts.population import master_catalog, master_catalog_factor
from sorts.propagator import SGP4
from sorts.space_object import SpaceObject
from sorts.radar.radars import get_radar
from sorts.types import Datetime64_us, Timedelta64_us, Float64_as_sec
from sorts.utils import to_datetime64_us
from sorts.controller_v2.tracker_controller import TrackerController
from sorts.controller_v2.fence_scan_controller_new import FenceScanController
from sorts.schedule_v2 import Schedule, ExperimentDetail
from sorts.scheduler_v2.priority_scheduling import priority_scheduling

# import for plottings
from IPython.display import display
import pandas as pd
import ipywidgets as widgets
from sorts.plotting_deps import lp, bp, bokeh_models, pn
from sorts import plots

The geodata is provided by © OpenStreetMap contributors and is made available here under the Open Database License (ODbL).


In [2]:
# disable pandas table wrapping
pd.set_option("display.expand_frame_repr", False)

# activate Bokeh output in Jupyter notebook
from bokeh.io import output_notebook, push_notebook
output_notebook()

# config and init lets-plot
lp.LetsPlot.setup_html()

# config and init panel
pn.extension("tabulator", comms='ipywidgets',
    # sizing_mode="stretch_width",
)

Loading BokehJS ...

In [3]:
epoch = Time(53005.0, format="mjd", scale="utc")  # 2004-01-01 00:00:00Z

# the first set of value used, not much use now; kept for ref
# start_time = Time("2025-06-30 00:00:00")
# end_time = Time("2025-06-30 00:00:01")
# control_slice_duration = np.timedelta64(10_000, "us")  # 10ms

# a 1 sec long period, the sbobj should be very close to right up ahead of eiscat3d tx-0 station
# start_time = Time("2025-01-01 04:04:00")
# end_time = Time("2025-01-01 04:04:01")
# control_slice_duration = np.timedelta64(10_000, "us")  # 10ms

# an extended duration which expands around from the 1 sec period above
# the `control_slice_duration` is much longer than normal, practical radar `control_slice_duration`
# for easier debugging, inspection of scheduling/schedules
# start_time = Time("2025-01-01 02:45:00")
# end_time = Time("2025-01-01 06:15:00")
# control_slice_duration = np.timedelta64(int(60 * 1e6), "us")

# same as above, but use more realistic 10ms `control_slice_duration`
start_time = Time("2025-01-01 02:45:00")
end_time = Time("2025-01-01 06:15:00")
control_slice_duration = np.timedelta64(10_000, "us")  # 10ms

eiscat3d = get_radar("eiscat3d", "stage1-array")

tracked_spobj = SpaceObject(
    SGP4,
    propagator_options={"settings": {"out_frame": "ITRF"}},
    a=7200e3,
    e=0.02,
    i=75,
    raan=86,
    aop=0,
    mu0=60,
    epoch=epoch,
    parameters={"d": 0.1},
)

catalog_fpath = Path() / ".." /  ".." / "local_data" / "celn_20090501_00.sim"
_spobj_pop = master_catalog(
    catalog_fpath,
    propagator=SGP4,
    propagator_options={"settings": {"in_frame": "TEME", "out_frame": "ITRF"}},
)
rand_seed = 120389
# TODO: reduce the filter size to more sensible value
spobj_pop = master_catalog_factor(_spobj_pop, treshhold=5.0, seed=rand_seed)
spobjs = [tracked_spobj, *[spobj_pop.get_object(i) for i in range(spobj_pop.shape[0])]]

exp_detail_0 = ExperimentDetail(
    id=0,
    coh_int_bandwidth=1.0,
    ipp=1.0,
    pulse_length=1.0,
    power=5000000.0,
    bandwidth=52.08333333333333,
    duty_cycle=1.0,
    noise_temp=150.0,
    slice_duration=control_slice_duration
)

exp_detail_1 = ExperimentDetail(
    id=1,
    coh_int_bandwidth=1.0,
    ipp=1.0,
    pulse_length=1.0,
    power=5000000.0,
    bandwidth=52.08333333333333,
    duty_cycle=1.0,
    noise_temp=150.0,
    slice_duration=control_slice_duration
)

time_arr: npt.NDArray[Datetime64_us] = np.arange(
    to_datetime64_us(start_time),
    to_datetime64_us(end_time),
    control_slice_duration,
)
# time_arr = time_arr[::4] # TODO: remove; strided to bring up the effects of scheduling
dt_arr: npt.NDArray[Timedelta64_us] = time_arr - to_datetime64_us(epoch)
dsec_arr: npt.NDArray[Float64_as_sec] = dt_arr.astype(np.float64) / 1e6  # type: ignore

ecefs = tracked_spobj.get_state(dsec_arr)

trackerController = TrackerController(
    tx_station=eiscat3d.tx[0],
    rx_stations=[],
    time=time_arr,
    space_object_states=ecefs,
    exp_detail=exp_detail_0,
    min_elevation=10,
)

fenceScanController = FenceScanController(
    tx_station=eiscat3d.tx[0],
    rx_station=[],
    exp_datail=exp_detail_1,
    azimuth=90, # sweep from east to west
    min_elevation=30,
    pointings_per_cycle=40,
)

In [4]:
data_table = plots.space_object_population_table_plot(spobj_pop)

bp.show(data_table)

In [5]:
plot = plots.kepler_space_object_on_map(spobjs[0], epoch, start_time=start_time)
plot.show()

In [6]:
# plot = plots.kepler_space_object_on_map(spobjs[4], epoch, start_time=start_time) # interesting s shape
# plot = plots.kepler_space_object_on_map(spobjs[5], epoch, start_time=start_time) # show be visible to eiscat
plot = plots.kepler_space_object_on_map(spobjs[17], epoch, start_time=start_time) # show be visible to eiscat
plot.show()

/home/erich/projects/irf/sorts/.venv/lib/python3.10/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "taiutc" yielded 500 of "dubious year (Note 4)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
/home/erich/projects/irf/sorts/.venv/lib/python3.10/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "utcut1" yielded 500 of "dubious year (Note 3)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
/home/erich/projects/irf/sorts/.venv/lib/python3.10/site-packages/astropy/coordinates/builtin_frames/utils.py:65: AstropyWarning: Tried to get polar motions for times after IERS data is valid. Defaulting to polar motion from the 50-yr mean for those. This may affect precision at the arcsec level. Please check your astropy.utils.iers.conf.iers_auto_url and point it to a newer version if necessary.
  warnings.warn(wmsg.format("after"), AstropyWarning)
/home/erich/projects/irf/sorts/.venv/lib/python3.10/site-packages/erfa/core.py:133: ErfaWarnin

In [7]:
# plots.ecef_states_positions_plot(ecefs)

In [8]:
tracker_schs = trackerController.generate()
tracker_tx_sch_df = tracker_schs.tx_schedule.as_dataframe()
tracker_tx_sch_df

,start_time,pointing_az,pointing_el,exp_num,end_time
0,2025-01-01 02:45:00.000,17.942607,14.412365,0,2025-01-01 02:45:00.010
1,2025-01-01 02:45:00.010,17.942534,14.411811,0,2025-01-01 02:45:00.020
2,2025-01-01 02:45:00.020,17.942462,14.411258,0,2025-01-01 02:45:00.030
3,2025-01-01 02:45:00.030,17.942389,14.410704,0,2025-01-01 02:45:00.040
4,2025-01-01 02:45:00.040,17.942317,14.410150,0,2025-01-01 02:45:00.050
...,...,...,...,...,...
563758,2025-01-01 06:12:42.670,63.279783,10.002758,0,2025-01-01 06:12:42.680
563759,2025-01-01 06:12:42.680,63.279859,10.002204,0,2025-01-01 06:12:42.690
563760,2025-01-01 06:12:42.690,63.279934,10.001650,0,2025-01-01 06:12:42.700
563761,2025-01-01 06:12:42.700,63.280009,10.001096,0,2025-01-01 06:12:42.710


In [9]:
# check schedule df memory usage (MB)
tracker_tx_sch_df.memory_usage().sum()/1e6

np.float64(22.550648)

In [10]:
fence_schs = fenceScanController.generate(start_time, end_time)
fence_tx_sch_df = fence_schs.tx_schedule.as_dataframe()
fence_tx_sch_df

,start_time,pointing_az,pointing_el,exp_num,end_time
0,2025-01-01 02:45:00.000,90.0,30.000000,1,2025-01-01 02:45:00.010
1,2025-01-01 02:45:00.010,90.0,33.076923,1,2025-01-01 02:45:00.020
2,2025-01-01 02:45:00.020,90.0,36.153846,1,2025-01-01 02:45:00.030
3,2025-01-01 02:45:00.030,90.0,39.230769,1,2025-01-01 02:45:00.040
4,2025-01-01 02:45:00.040,90.0,42.307692,1,2025-01-01 02:45:00.050
...,...,...,...,...,...
1259995,2025-01-01 06:14:59.950,270.0,42.307692,1,2025-01-01 06:14:59.960
1259996,2025-01-01 06:14:59.960,270.0,39.230769,1,2025-01-01 06:14:59.970
1259997,2025-01-01 06:14:59.970,270.0,36.153846,1,2025-01-01 06:14:59.980
1259998,2025-01-01 06:14:59.980,270.0,33.076923,1,2025-01-01 06:14:59.990


In [11]:
# check schedule df memory usage (MB)
fence_tx_sch_df.memory_usage().sum()/1e6

np.float64(50.400128)

In [12]:
tx_sch = tracker_schs.tx_schedule
# plots.azel_polar_plot(tx_sch.pointing_az, tx_sch.pointing_el)

In [13]:
tx_sch = fence_schs.tx_schedule
# plots.azel_polar_plot(tx_sch.pointing_az, tx_sch.pointing_el)

In [14]:
(tracker_schs.tx_schedule.meta, fence_schs.tx_schedule.meta)

({0: ExperimentDetail(id=0, coh_int_bandwidth=1.0, ipp=1.0, pulse_length=1.0, power=5000000.0, bandwidth=52.08333333333333, duty_cycle=1.0, noise_temp=150.0, slice_duration=np.timedelta64(10000,'us'))},
 {1: ExperimentDetail(id=1, coh_int_bandwidth=1.0, ipp=1.0, pulse_length=1.0, power=5000000.0, bandwidth=52.08333333333333, duty_cycle=1.0, noise_temp=150.0, slice_duration=np.timedelta64(10000,'us'))})

In [15]:
master_sch = priority_scheduling([tracker_schs.tx_schedule, fence_schs.tx_schedule], {0: exp_detail_0, 1: exp_detail_1})
master_sch

Schedule(meta={0: ExperimentDetail(id=0, coh_int_bandwidth=1.0, ipp=1.0, pulse_length=1.0, power=5000000.0, bandwidth=52.08333333333333, duty_cycle=1.0, noise_temp=150.0, slice_duration=np.timedelta64(10000,'us')), 1: ExperimentDetail(id=1, coh_int_bandwidth=1.0, ipp=1.0, pulse_length=1.0, power=5000000.0, bandwidth=52.08333333333333, duty_cycle=1.0, noise_temp=150.0, slice_duration=np.timedelta64(10000,'us'))}, start_time=array(['2025-01-01T02:45:00.000000', '2025-01-01T02:45:00.010000',
       '2025-01-01T02:45:00.020000', ..., '2025-01-01T06:12:42.690000',
       '2025-01-01T06:12:42.700000', '2025-01-01T06:12:42.710000'],
      shape=(1246268,), dtype='datetime64[us]'), exp_num=array([0, 0, 0, ..., 0, 0, 0], shape=(1246268,)), pointing_az=array([17.94260693, 17.94253442, 17.94246191, ..., 63.27993392,
       63.28000913, 63.28008434], shape=(1246268,)), pointing_el=array([14.41236506, 14.41181136, 14.41125769, ..., 10.00164985,
       10.0010957 , 10.00054155], shape=(1246268,)))

In [16]:
master_sch_df = master_sch.as_dataframe()
master_sch_df

,start_time,pointing_az,pointing_el,exp_num,end_time
0,2025-01-01 02:45:00.000,17.942607,14.412365,0,2025-01-01 02:45:00.010
1,2025-01-01 02:45:00.010,17.942534,14.411811,0,2025-01-01 02:45:00.020
2,2025-01-01 02:45:00.020,17.942462,14.411258,0,2025-01-01 02:45:00.030
3,2025-01-01 02:45:00.030,17.942389,14.410704,0,2025-01-01 02:45:00.040
4,2025-01-01 02:45:00.040,17.942317,14.410150,0,2025-01-01 02:45:00.050
...,...,...,...,...,...
1246263,2025-01-01 06:12:42.670,63.279783,10.002758,0,2025-01-01 06:12:42.680
1246264,2025-01-01 06:12:42.680,63.279859,10.002204,0,2025-01-01 06:12:42.690
1246265,2025-01-01 06:12:42.690,63.279934,10.001650,0,2025-01-01 06:12:42.700
1246266,2025-01-01 06:12:42.700,63.280009,10.001096,0,2025-01-01 06:12:42.710


In [17]:
master_sch_df[master_sch_df["exp_num"] == 1]

,start_time,pointing_az,pointing_el,exp_num,end_time
7965,2025-01-01 02:46:19.660,90.0,48.461538,1,2025-01-01 02:46:19.670
7966,2025-01-01 02:46:19.670,90.0,51.538462,1,2025-01-01 02:46:19.680
7967,2025-01-01 02:46:19.680,90.0,54.615385,1,2025-01-01 02:46:19.690
7968,2025-01-01 02:46:19.690,90.0,57.692308,1,2025-01-01 02:46:19.700
7969,2025-01-01 02:46:19.700,90.0,60.769231,1,2025-01-01 02:46:19.710
...,...,...,...,...,...
966860,2025-01-01 05:26:08.630,270.0,79.230769,1,2025-01-01 05:26:08.640
966861,2025-01-01 05:26:08.640,270.0,76.153846,1,2025-01-01 05:26:08.650
966862,2025-01-01 05:26:08.650,270.0,73.076923,1,2025-01-01 05:26:08.660
966863,2025-01-01 05:26:08.660,270.0,70.000000,1,2025-01-01 05:26:08.670


In [20]:
# plots.schedule_plot(master_sch)

In [21]:
schedule = master_sch

df = schedule.as_dataframe()

start_datetime_widget = widgets.DatetimePicker(
    value=df[schedule.cn.start_time].min().tz_localize("utc"),
    description='Start Time',
)
end_datetime_widget = widgets.DatetimePicker(
    value=df[schedule.cn.start_time].min().tz_localize("utc") + np.timedelta64(5, "m"),
    description='End Time',
)

date_range_widget = widgets.HBox([start_datetime_widget, end_datetime_widget])
date_range_widget

In [22]:
# TODO: leverage `notebook_handle`, e.g. `plot_nbh = bp.show(plot, notebook_handle=True)` ?
plot = plots.schedule_plot_bokeh(master_sch, start_datetime_widget.value.replace(tzinfo=None), end_datetime_widget.value.replace(tzinfo=None))
bp.show(plot)